# Colab Setup

Mount Google Drive and pull the latest repo changes before running the experiment cells.

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted


In [2]:
import subprocess
from pathlib import Path

REPO_PATH = Path('/content/drive/MyDrive/repos/evolutionary-ai-battle')

if REPO_PATH.exists():
    result = subprocess.run(['git', 'pull'], cwd=REPO_PATH, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
else:
    print(f'Repo path not found: {REPO_PATH}')

Already up to date.



In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = REPO_PATH if 'REPO_PATH' in globals() and REPO_PATH.exists() else Path.cwd()
EXPERIMENT_ROOT = PROJECT_ROOT / 'experiment' if PROJECT_ROOT.name != 'experiment' else PROJECT_ROOT

for path in (PROJECT_ROOT, EXPERIMENT_ROOT):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
    print('autoreload enabled')
except Exception as exc:
    print('autoreload skipped:', exc)

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('DEVICE:', DEVICE)
    if DEVICE == 'cuda':
        print('GPU:', torch.cuda.get_device_name(0))
        print('CUDA version:', torch.version.cuda)
    else:
        print('No GPU is visible to this runtime. In Colab, use Runtime > Change runtime type > GPU, then rerun setup.')
except Exception as exc:
    DEVICE = 'cpu'
    print('torch device check skipped:', exc)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('EXPERIMENT_ROOT:', EXPERIMENT_ROOT)

autoreload skipped: No module named 'imp'
DEVICE: cuda
GPU: Tesla T4
CUDA version: 12.8
PROJECT_ROOT: /content/drive/MyDrive/repos/evolutionary-ai-battle
EXPERIMENT_ROOT: /content/drive/MyDrive/repos/evolutionary-ai-battle/experiment


# CPC TorchRL Scaffold

This notebook is the initiating experiment entry point. Core logic lives in Python modules; the notebook demonstrates the toy CPC loop, the TorchRL wrapper, and PPO smoke training.

In [4]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path.cwd()
EXPERIMENT_ROOT = EXPERIMENT_ROOT if 'EXPERIMENT_ROOT' in globals() else PROJECT_ROOT / "experiment"

for path in (PROJECT_ROOT, EXPERIMENT_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from training.cpc_actions import action_space, decode_action, describe_action
from training.cpc_env import CPCEnv

print("experiment root:", EXPERIMENT_ROOT)
print("action space:", action_space())

experiment root: /content/drive/MyDrive/repos/evolutionary-ai-battle/experiment
action space: ActionSpace(move_bins=9, aim_bins=16, fire_bins=2)


## PR1: Toy CPC Loop

In [5]:
env = CPCEnv(seed=7, max_steps=8)
obs = env.reset()
move_and_fire = {"move": 6, "aim": 0, "fire": 1}

print(json.dumps(describe_action(move_and_fire), indent=2))
obs, reward, done, info = env.step(move_and_fire)
print("reward", reward, "done", done)
print("decoded", info["decoded_action"])

for _ in range(3):
    action = env.sample_action()
    obs, reward, done, info = env.step(action)
    print({"raw": action, "reward": round(reward, 3), "done": done})
    if done:
        break

print("metrics", json.dumps(env.metrics.summary(), indent=2))
print("trajectory step", json.dumps(env.export_trajectory()["steps"][0], indent=2))

{
  "rawAction": {
    "move": 6,
    "aim": 0,
    "fire": 1
  },
  "moveLabel": "up_right",
  "decoded": {
    "moveX": 0.7071067811865475,
    "moveY": -0.7071067811865475,
    "aimX": 1.0,
    "aimY": 0.0,
    "fire": 1
  }
}
reward 0.02 done False
decoded {'moveX': 0.7071067811865475, 'moveY': -0.7071067811865475, 'aimX': 1.0, 'aimY': 0.0, 'fire': 1}
{'raw': {'move': 5, 'aim': 4, 'fire': 1}, 'reward': 0.02, 'done': False}
{'raw': {'move': 0, 'aim': 2, 'fire': 0}, 'reward': 0.02, 'done': False}
{'raw': {'move': 5, 'aim': 1, 'fire': 0}, 'reward': 0.02, 'done': False}
metrics {
  "avg_ally_distance": 162.45319070603566,
  "isolation_rate": 0.0,
  "teammate_under_pressure_response": 0.0,
  "damage_dealt": 0.0,
  "damage_taken": 0.0
}
trajectory step {
  "step": 1,
  "actorId": "self",
  "action": {
    "moveX": 0.7071067811865475,
    "moveY": -0.7071067811865475,
    "aimX": 1.0,
    "aimY": 0.0,
    "fire": 1
  },
  "rawAction": {
    "move": 6,
    "aim": 0,
    "fire": 1
  },
  "s

## PR2: TorchRL EnvBase / TensorDict Wrapper

In [6]:
try:
    import torch
    from training.torchrl_env import TorchRLCPCEnv
    from training.torchrl_specs import import_check_env_specs

    torchrl_env = TorchRLCPCEnv(seed=11, max_steps=8, device=DEVICE)
    print("observation_spec")
    print(torchrl_env.observation_spec)
    print("action_spec")
    print(torchrl_env.action_spec)

    td = torchrl_env.reset()
    td.update(torchrl_env.action_spec.rand())
    stepped = torchrl_env.step(td)
    print("sample reward", stepped["next", "reward"], "done", stepped["next", "done"])

    move_fire_td = torchrl_env.reset()
    move_fire_td["move"] = torch.tensor(6, dtype=torch.int64)
    move_fire_td["aim"] = torch.tensor(0, dtype=torch.int64)
    move_fire_td["fire"] = torch.tensor(1, dtype=torch.int64)
    move_fire_next = torchrl_env.step(move_fire_td)
    print("decoded engine action", move_fire_next["next", "decoded_action"])

    check_env_specs = import_check_env_specs()
    if check_env_specs is not None:
        check_env_specs(torchrl_env)
        print("check_env_specs passed")
except Exception as exc:
    print("TorchRL wrapper demo skipped:", exc)

TorchRL wrapper demo skipped: No module named 'tensordict'


# PR3: PPO Smoke Training

This section proves the adapter can feed a minimal PPO loop. It is smoke validation, not policy-performance work.

In [7]:
try:
    import torch
    from training.ppo_policy import MultiDiscreteActorCritic
    from training.train_ppo import PPOConfig, collect_rollout, train_ppo
    from training.eval_ppo import eval_checkpoint
    from training.torchrl_env import TorchRLCPCEnv

    env = TorchRLCPCEnv(seed=5, max_steps=8, device=DEVICE)
    policy = MultiDiscreteActorCritic(hidden_dim=32).to(DEVICE)
    obs = env.reset()
    output = policy.sample_action(obs)
    print("sampled policy action", {k: int(v) for k, v in output.action.items()})
    print("log_prob", float(output.log_prob.reshape(-1)[0]), "value", float(output.value.reshape(-1)[0]))

    rollout = collect_rollout(env, policy, PPOConfig(device=DEVICE, rollout_steps=8, max_episode_steps=8, hidden_dim=32))
    print("rollout obs shape", rollout["observations"].shape)

    cfg = PPOConfig(device=DEVICE, total_steps=32, rollout_steps=16, num_epochs=1, minibatch_size=8, max_episode_steps=8, hidden_dim=32)
    result = train_ppo(cfg)
    print("train result")
    print(json.dumps(result, indent=2))

    report = eval_checkpoint(result["checkpoint"], episodes=2)
    print("eval report")
    print(json.dumps(report, indent=2))
except Exception as exc:
    print("PPO smoke demo skipped:", exc)

# TODO PR4: scenario GT evaluation, trajectory export, and richer CPC metric reports.

PPO smoke demo skipped: attempted relative import with no known parent package
